# Tool-Using Agent — Direct Tool Dispatch

## What You'll Learn
- The **ToolAgent** pattern: how an LLM picks and calls tools in a single pass (no explicit reasoning loop)
- How tools work under the hood — schema generation, dispatch, and result handling
- How the **calculator** tool uses Python's `ast` module for safe expression evaluation
- How to inspect the full tool inventory available to a ToolAgent

## Prerequisites
```bash
pip install agentexplorr[agents]
# or: uv sync --extra agents
```

## Learning Resources
- [LangChain Tool Calling](https://python.langchain.com/docs/concepts/tool_calling/) — how `@tool` and `bind_tools()` work
- [LangChain Custom Tools](https://python.langchain.com/docs/how_to/custom_tools/) — building your own tools
- [Ollama Function Calling](https://ollama.com/blog/tool-support) — local tool-calling models
- [Video: Function Calling with LangChain](https://www.youtube.com/watch?v=p9v2fHLmQCU)
- [Video: AI Agents Tool Use Explained](https://www.youtube.com/watch?v=cN9S6CYjmi8)

In [ ]:
# Step 1: Import the safe evaluator and the LangChain calculator tool
from agentexplorr.agents.tools.calculator import safe_evaluate, calculator

print("Imports successful!")
print(f"  safe_evaluate : direct Python function -> returns numeric types")
print(f"  calculator    : LangChain @tool wrapper -> returns formatted strings")
print()
print(f"Calculator tool name:        {calculator.name}")
print(f"Calculator tool description: {calculator.description[:80]}...")

## The Calculator Tool — Safe Math via AST Parsing

Most calculator tools use Python's `eval()`, which is **dangerously insecure** — an LLM could generate `__import__('os').system('rm -rf /')` and `eval()` would execute it.

AgentExplorr's calculator uses a much safer approach based on the **Abstract Syntax Tree (AST)**:

1. **Parse** the expression into an AST with `ast.parse()`
2. **Validate** every node against a whitelist (numbers, arithmetic ops, math functions)
3. **Reject** anything not whitelisted — imports, attribute access, function defs — *before* execution
4. **Evaluate** only the safe nodes using a restricted namespace of math functions

This whitelist-based approach means new Python features cannot accidentally become exploitable.

### Supported operations
| Category | Examples |
|----------|---------|
| Arithmetic | `+`, `-`, `*`, `/`, `//`, `%`, `**` |
| Functions | `sqrt`, `sin`, `cos`, `log`, `log10`, `factorial`, `ceil`, `floor` |
| Constants | `pi`, `e`, `tau`, `inf` |
| Aggregation | `min`, `max`, `sum` (with list literals) |

In [ ]:
# Step 2: Demonstrate safe_evaluate with various expressions
# safe_evaluate returns native Python types (int, float, bool)

expressions = [
    ("Basic arithmetic",     "2 + 3"),
    ("Order of operations",  "2 + 3 * 4"),
    ("Exponentiation",       "2 ** 10"),
    ("Square root + pi",     "sqrt(144) + pi"),
    ("Factorial",            "factorial(7)"),
    ("Trigonometry",         "sin(pi / 2)"),
    ("Logarithm base 10",   "log10(1000)"),
    ("Natural log",          "log(e ** 5)"),
    ("Floor division",       "17 // 3"),
    ("Comparison",           "pi > 3"),
]

print("safe_evaluate() demo — AST-based safe math evaluation")
print("=" * 55)
for label, expr in expressions:
    result = safe_evaluate(expr)
    print(f"  {label:.<25s} {expr:>20s}  =  {result}")

# Now show that unsafe expressions are rejected
print()
print("Security demo — unsafe expressions are blocked:")
print("-" * 55)
from agentexplorr.agents.tools.calculator import SafeExpressionError

unsafe_expressions = [
    "__import__('os')",
    "open('/etc/passwd')",
    "lambda x: x",
]
for expr in unsafe_expressions:
    try:
        safe_evaluate(expr)
    except SafeExpressionError as e:
        print(f"  BLOCKED: {expr:30s} -> {e}")

## How the ToolAgent Architecture Works

The **ToolAgent** is the simplest agent pattern in AgentExplorr. Unlike the ReAct agent (which has an explicit Think/Act/Observe loop), the ToolAgent lets the LLM handle reasoning implicitly:

```
┌─────────┐
│  START   │
└────┬─────┘
     │
     ▼
┌────────────┐   tool_calls   ┌────────────┐
│ call_model  │──────────────→│ call_tools  │
└─────┬──────┘                └──────┬──────┘
      │                               │
      │ no tool_calls                 │ (loop back)
      ▼                               │
 ┌─────────┐                          │
 │   END   │  ◄───────────────────────┘
 └─────────┘
```

### How tool dispatch works under the hood

1. **Schema generation** — The `@tool` decorator extracts each function's name, docstring, and type hints into a JSON schema that the LLM can read.
2. **Tool binding** — `llm.bind_tools(tools)` attaches the tool schemas to the LLM so it knows what tools are available.
3. **Structured output** — The LLM produces a structured response indicating which tool to call and with what arguments.
4. **Dispatch** — The framework parses the LLM's output, looks up the tool by name in `_TOOL_MAP`, and calls `tool.invoke(args)`.
5. **Result loop** — The tool's return value is sent back to the LLM as a `ToolMessage`. The LLM can then call another tool or produce a final answer.

### Tool-Calling vs. ReAct — when to use which

| Aspect | Tool-Calling | ReAct |
|--------|-------------|-------|
| Complexity | Simple (1-2 tool calls) | Multi-step reasoning chains |
| Reasoning | Implicit (inside the LLM) | Explicit (visible THINK step) |
| Latency | Lower (fewer LLM calls) | Higher (multiple rounds) |
| Best for | "What is sqrt(144)?" | "Compare GDP of 3 countries" |

In [ ]:
# Step 3: Explore the full tool inventory available to the ToolAgent
#
# The ToolAgent ships with 6 built-in tools. Each tool is a LangChain @tool
# decorated function with a name, description, and typed arguments.
# The LLM reads these descriptions to decide which tool to call.

from agentexplorr.agents.tool_agent import TOOL_AGENT_TOOLS

print(f"ToolAgent has {len(TOOL_AGENT_TOOLS)} tools available:\n")
print(f"{'#':<4} {'Name':<22} {'Description'}")
print("-" * 80)
for i, t in enumerate(TOOL_AGENT_TOOLS, 1):
    # Each tool's description comes from its docstring (first line)
    desc = t.description.split("\n")[0]
    print(f"{i:<4} {t.name:<22} {desc}")

# Show the JSON schema that the LLM actually sees for one tool
print("\n" + "=" * 80)
print("Example: JSON schema the LLM receives for 'calculator':")
print("=" * 80)
import json
print(json.dumps(calculator.args_schema.schema(), indent=2))

## Key Takeaways

1. **Tool-calling agents are the simplest agent pattern** — the LLM decides which tool to call in a single pass, with no explicit reasoning loop.
2. **Safety matters** — the calculator uses AST-based parsing instead of `eval()` to prevent code injection attacks. Always sanitize LLM-generated inputs before execution.
3. **Tools are just decorated functions** — the `@tool` decorator converts a Python function's name, docstring, and type hints into a JSON schema that the LLM can understand and invoke.
4. **Tool design drives agent quality** — clear tool names and descriptive docstrings help the LLM pick the right tool. The docstring *is* the tool's documentation for the LLM.
5. **`_TOOL_MAP` enables dispatch** — a dictionary mapping tool names to callable objects lets the framework look up and execute tools by the name the LLM returns.

## Try It Yourself
```python
# Run the full ToolAgent (requires Ollama with llama3.2):
#   ollama pull llama3.2 && ollama serve
from agentexplorr.agents.tool_agent import ToolAgent
agent = ToolAgent(verbose=True)
result = agent.run("What is the square root of 256 plus pi?")
print(result.answer)
```

## Next Steps
- Try the [Multi-Agent System notebook](03_multi_agent_system.ipynb) to see how multiple agents collaborate
- Read the source: `src/agentexplorr/agents/tool_agent.py`
- Read about tool design: `src/agentexplorr/agents/tools/__init__.py`
- [LangGraph Tool Calling How-To](https://langchain-ai.github.io/langgraph/how-tos/tool-calling/)